# Module 2

In [1]:
# import AWS SDK for Python
import boto3

# import json and sys
import json
import sys

## Listing Foundation Models

In [2]:

# Instantiate a bedrock client
bedrock = boto3.client("bedrock", region_name="eu-north-1")
# List foundation models
response = bedrock.list_foundation_models()

try:
    # Get the list of foundation models
    response = bedrock.list_foundation_models()

    # Extract only the useful model information
    models = response.get("modelSummaries", [])

    if not models:
        print("No foundation models found.")
    else:
        print(f"Found {len(models)} foundation models:\n")

        for model in models:
            print(f"Model Name   : {model.get('modelName', 'N/A')}")
            print(f"Model ID     : {model.get('modelId', 'N/A')}")
            print(f"Provider     : {model.get('providerName', 'N/A')}")
            print(f"Input        : {', '.join(model.get('inputModalities', []))}")
            print(f"Output       : {', '.join(model.get('outputModalities', []))}")
            print(f"Inference    : {', '.join(model.get('inferenceTypesSupported', []))}")
            print("-" * 70)

except Exception as e:
    print(f"Error while listing Bedrock foundation models: {e}")

Found 42 foundation models:

Model Name   : Claude Opus 5
Model ID     : anthropic.claude-opus-5
Provider     : Anthropic
Input        : TEXT, IMAGE
Output       : TEXT
Inference    : INFERENCE_PROFILE
----------------------------------------------------------------------
Model Name   : GPT-6 Astra
Model ID     : openai.gpt-6-astra
Provider     : OpenAI
Input        : TEXT, IMAGE
Output       : TEXT
Inference    : INFERENCE_PROFILE
----------------------------------------------------------------------
Model Name   : GPT-5.6 Terra
Model ID     : openai.gpt-5.6-terra
Provider     : OpenAI
Input        : TEXT, IMAGE
Output       : TEXT
Inference    : INFERENCE_PROFILE
----------------------------------------------------------------------
Model Name   : Claude Sonnet 4
Model ID     : anthropic.claude-sonnet-4-20250514-v1:0
Provider     : Anthropic
Input        : TEXT, IMAGE
Output       : TEXT
Inference    : INFERENCE_PROFILE
----------------------------------------------------------------

## Invoking models using invokeModel

In [3]:

# Instantiate a bedrock-runtime client in us-east-1
bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

### Llama 3's instruct model

In [7]:
# Embed the prompt in Llama 3's instruction format.
prompt = "Describe the purpose of a 'hello world' program in one line."
formatted_prompt = f"""
<|begin_of_text|><|start_header_id|>user<|end_header_id|>
{prompt}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

# Invoke the model
response = bedrock_runtime.invoke_model(
    modelId="meta.llama3-70b-instruct-v1:0", 
    body=json.dumps({
        "prompt": formatted_prompt,
        "max_gen_len": 512,
        "temperature": 0.5,
    })
)

# Decode the response body and print it
model_response = json.loads(response["body"].read())
print(json.dumps(model_response, indent=2))

{
  "generation": "The purpose of a \"Hello World\" program is to verify that a programming language, compiler, or development environment is correctly installed and configured by printing a simple message to the screen.",
  "prompt_token_count": 26,
  "generation_token_count": 37,
  "stop_reason": "stop"
}


### Mistral AI

In [8]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."
formatted_prompt = f"<s>[INST]{prompt}[/INST]"

# Invoke the model
response = bedrock_runtime.invoke_model(
    modelId="mistral.mistral-large-2402-v1:0", 
    body=json.dumps({
        "prompt": formatted_prompt,
        "max_tokens": 512,
        "temperature": 0.5,
    })
)

# Decode the response body and print it
model_response = json.loads(response["body"].read())
print(json.dumps(model_response, indent=2))

{
  "outputs": [
    {
      "text": " The purpose of a 'hello world' program is to provide a simple introduction to a programming language, demonstrating its basic syntax and confirming that the programming environment is working correctly.",
      "stop_reason": "stop"
    }
  ]
}


### Amazon Nova

In [ ]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Invoke the model
response = bedrock_runtime.invoke_model(
    modelId="amazon.nova-lite-v1:0", 
    body=json.dumps({
        "inferenceConfig": {
            "maxTokens": 512,
            "temperature": 0.5
        },
        "messages": [
            {
                "role": "user",
                "content": [{
                    "text": prompt
                }]
            }
        ]
    })
)

# Decode the response body and print it
model_response = json.loads(response["body"].read())
print(json.dumps(model_response, indent=2))

## Converse API

### Llama 3's instruct model

In [12]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Invoke the model using converse
response = bedrock_runtime.converse(
    modelId="meta.llama3-70b-instruct-v1:0", 
    inferenceConfig = {
        "maxTokens": 512,
        "temperature": 0.5
    },
    messages = [
        {
            "role": "user",
            "content": [{
                "text": prompt
            }]
        }
    ]
)

# Print the response output
print(json.dumps(response, indent=2))

{
  "ResponseMetadata": {
    "RequestId": "fe5557ba-92c2-48b6-893a-b6df327a5f19",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Thu, 03 Sep 2026 16:36:32 GMT",
      "content-type": "application/json",
      "content-length": "416",
      "connection": "keep-alive",
      "x-amzn-requestid": "fe5557ba-92c2-48b6-893a-b6df327a5f19"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "\n\nA \"hello world\" program is a simple program that outputs \"Hello, World!\" to demonstrate the basic syntax and functionality of a programming language, serving as a introductory exercise for new programmers."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 28,
    "outputTokens": 40,
    "totalTokens": 68
  },
  "metrics": {
    "latencyMs": 1100
  }
}


### Mistral AI

In [ ]:
import boto3
from botocore.exceptions import ClientError

bedrock_runtime = boto3.client("bedrock-runtime")

# Model
model_id = "us.anthropic.claude-3-haiku-20240307-v1:0"

# Prompt
prompt = "Describe the purpose of a 'hello world' program in one line."

# Conversation structure
messages = [
    {
        "role": "user",
        "content": [
            {
                "text": prompt
            }
        ]
    }
]

try:
    response = bedrock_runtime.converse(
        modelId=model_id,

        messages=messages,

        # Parameters supported directly by Converse API
        inferenceConfig={
            "maxTokens": 512,
            "temperature": 0.5,
            "topP": 0.9
        },

        # Model-specific Anthropic parameter
        additionalModelRequestFields={
            "top_k": 50
        }
    )

    # Extract only the generated text
    output_text = response["output"]["message"]["content"][0]["text"]

    print("Model response:")
    print(output_text)

except ClientError as e:
    print(f"AWS Bedrock error: {e}")

except Exception as e:
    print(f"Error: {e}")

AWS Bedrock error: An error occurred (ResourceNotFoundException) when calling the Converse operation: Access denied. This Model is marked by provider as Legacy and you have not been actively using the model in the last 30 days. Please upgrade to an active model on Amazon Bedrock


### Amazon Nova

In [ ]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Invoke the model using converse
response = bedrock_runtime.converse(
    modelId="us.amazon.nova-lite-v1:0", 
    inferenceConfig = {
            "maxTokens": 512,
            "temperature": 0.5,
            "topP": 0.9,
            "topk": 50
        },
    messages = [
        {
            "role": "user",
            "content": [{
                "text": prompt
            }]
        }
    ]
)

# Print the response output
print(json.dumps(response, indent=2))

## ConverseStream API
Showing the entire output

In [18]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in one line."

# Invoke the model using converse
response = bedrock_runtime.converse_stream(
    modelId="us.amazon.nova-lite-v1:0", 
    inferenceConfig = {
        "maxTokens": 512,
        "temperature": 0.5
    },
    messages = [
        {
            "role": "user",
            "content": [{
                "text": prompt
            }]
        }
    ]
)

for event in response["stream"]:
    print(json.dumps(event, indent=2))

{
  "messageStart": {
    "role": "assistant"
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": "The"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " purpose"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " of"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " a"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " '"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": "hello"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " world"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": "'"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDelta": {
    "delta": {
      "text": " program"
    },
    "contentBlockIndex": 0
  }
}
{
  "contentBlockDe

Showing only the text to see it as streamed

In [5]:
# Define the prompt for the model.
prompt = "Describe the purpose of a 'hello world' program in a text of at least 500 words"

# Invoke the model using converse
response = bedrock_runtime.converse_stream(
    modelId="amazon.nova-lite-v1:0", 
    inferenceConfig = {
        "maxTokens": 1024,
        "temperature": 0.5
    },
    messages = [
        {
            "role": "user",
            "content": [{
                "text": prompt
            }]
        }
    ]
)

for event in response["stream"]:
    if "contentBlockDelta" in event:
        chunk = event["contentBlockDelta"]
        sys.stdout.write(chunk["delta"]["text"])
        sys.stdout.flush()

A "Hello, World!" program is often the first step in learning a new programming language. It is a simple yet profound introduction to the basics of coding, syntax, and execution. The purpose of a "Hello, World!" program extends beyond its simplicity, serving as a foundational milestone for both novice and experienced programmers. This essay will explore the significance of the "Hello, World!" program, its role in the learning process, and its broader implications in the world of software development.

At its core, a "Hello, World!" program is designed to produce the output "Hello, World!" on the screen. This output is achieved by writing a few lines of code in a chosen programming language. Despite its simplicity, this program encapsulates several fundamental concepts of programming. It demonstrates how to write a program, how to compile or interpret it, and how to execute it to achieve the desired output. For beginners, this is often the first concrete interaction with a programming l